# 3. Forecast turnout for Zimran (5 Apr 2026)

A unified version of the model without age (the Zimran form does not ask for it). Coarser registration-timing groups (Last Minute / Late / Regular / Early Bird) and a broader major mapping let the same model score both the Creative One and Zimran forms. The last cell checks the accuracy of this setup on a Higgsfield validation split and prints feature importance.

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
import numpy as np

# ==========================================
# 1. UNIFIED MAPPING FUNCTIONS
# ==========================================
def group_major_unified(major):
    m = str(major).lower().strip()
    if any(x in m for x in ['data', 'ai', 'аналит', 'интеллект', 'intelligence', 'bda', 'bda', 'machine learning', 'ml']): return 'Data Science & AI'
    if any(x in m for x in ['comput', 'soft', 'cs', 'программ', 'информ', 'it', 'swe', 'ict', 'developer', 'smart tech', 'web', 'ios', 'information', 'backend']): return 'Computer Science'
    if any(x in m for x in ['cyber', 'безопасн', 'систем', 'security']): return 'IT & Cybersecurity'
    if any(x in m for x in ['engin', 'инженер', 'механ', 'строй', 'robot', 'робот', 'elce', 'ece', 'engin', 'инженер', 'механ', 'строй', 'robot', 'робот', 'elce', 'ece', 'mae', 'cee', 'chme', 'mining', 'civil', 'automation', 'автоматиз']): return 'Engineering'
    if any(x in m for x in ['econ', 'finan', 'accoun', 'эконом', 'финанс', 'учет', 'аудит', 'econ', 'finan', 'accoun', 'эконом', 'финанс', 'учет', 'аудит', 'бухгалтер', 'finance']): return 'Economics & Finance'
    if any(x in m for x in ['manag', 'marke', 'busin', 'менедж', 'маркет', 'бизнес', 'mba', 'bba', 'менеджмент', 'manag', 'marke', 'busin', 'менедж', 'маркет', 'бизнес', 'mba', 'bba', 'hr', 'logistics', 'логистика', 'tourism', 'туризм', 'smm', 'pr', 'предприн']): return 'Business & Management'
    if any(x in m for x in ['polit', 'relat', 'socio', 'psir', 'journ', 'psych', 'право', 'юрисп', 'филол', 'лингв', 'literature', 'law', 'лингвистика', 'журналистика', 'polit', 'relat', 'socio', 'psir', 'journ', 'psych', 'право', 'юрисп', 'филол', 'лингв', 'literature', 'law', 'лингвистика', 'журналистика', 'психолог', 'педагог', 'teacher', 'мұғалім', 'wll', 'translation', 'аударма', 'media', 'archi', 'дизайн', 'design', 'filmmaking', 'history', 'ориентал', 'oriental']): return 'Social Sciences & Humanities'
    if any(x in m for x in ['med', 'врач', 'health', 'pharma', 'биолог', 'biology', 'bio', 'медицина', 'стоматология', 'chemistry', 'хими', 'geology', 'геолог', 'physics', 'физик', 'nusom']): 
        return 'Medicine & Science'   
    if any(x in m for x in ['foundation', 'nufyp', 'foundation']): 
        return 'Foundation'    
    return 'Other'

def map_uni_unified(uni):
    uni = str(uni).lower()
    if any(x in uni for x in ['nazarbayev', 'назарбаев', 'nu']): return 'Nazarbayev University'
    if any(x in uni for x in ['astana it', 'aitu']): return 'Astana IT University'
    if any(x in uni for x in ['гумилев', 'enu', 'еуразия']): return 'Eurasian National University'
    if any(x in uni for x in ['narikbayev', 'kazguu', 'қазгу', 'mnu']): return 'Maqsut Narikbayev University'
    if any(x in uni for x in ['сейфуллин', 'kazatu']): return 'Kazakh Agrotechnical University'
    return 'Other'

def map_degree_unified(deg):
    deg = str(deg).lower()
    if '1' in deg: return '1st Year'
    if '2' in deg: return '2nd Year'
    if '3' in deg: return '3rd Year'
    if '4' in deg: return '4th Year'
    if 'master' in deg or 'магистрант' in deg: return "Master's"
    if any(x in deg for x in ['түлегі', 'выпускник', 'graduate']): return 'Graduate'
    return 'Other'

def get_reg_cat(days):
    if days <= 1: return "Last Minute"
    if days <= 3: return "Late"
    if days <= 7: return "Regular"
    return "Early Bird"

# Standard projects to track
projects_list = ['AIESEC Recruitment', 'Creative One', 'GO Exchanges', 'SheCodes', 'OZGE Forum', 'EmployMe']

# ==========================================
# 2. TRAIN MODEL ON HIGGSFIELD (NO AGE)
# ==========================================
print("Training model on Higgsfield dataset...")
df_h = pd.read_csv('data/Higgsfield_Registrations.csv')  # private, not included
h_date = pd.Timestamp('2025-10-18')

# Standardize Higgsfield
df_h['Degree'] = df_h['Your education degree:'].apply(map_degree_unified)
df_h['University'] = df_h['Your place of study:'].apply(map_uni_unified)
df_h['Major_Group'] = df_h['Your major:'].apply(group_major_unified)

# Reg Category
df_h['Submit Date (UTC)'] = pd.to_datetime(df_h['Submit Date (UTC)']).dt.tz_localize(None)
df_h['Reg_Day_Category'] = ((h_date - df_h['Submit Date (UTC)']).dt.days).apply(get_reg_cat)

# Projects (Higgs has them in separate columns)
for p in projects_list:
    df_h[p] = df_h[p].notna().astype(int)

# Select Features (Notice: AGE IS REMOVED)
features = ['Degree', 'University', 'Major_Group', 'Reg_Day_Category'] + projects_list
cat_features = ['Degree', 'University', 'Major_Group', 'Reg_Day_Category']

model = CatBoostClassifier(iterations=1000, depth=6, cat_features=cat_features, verbose=False)
model.fit(df_h[features], df_h['CheckIn'])

# ==========================================
# 3. PROCESS ZIMRAN DATA
# ==========================================
#print("\nProcessing Zimran Data...")
df_z = pd.read_csv('data/ZimranApril3.csv')  # private, not included
z_date = pd.Timestamp('2026-04-05')

df_z['Degree'] = df_z['year_of_study'].astype(str).apply(map_degree_unified)
df_z['University'] = df_z['university'].apply(map_uni_unified)
df_z['Major_Group'] = df_z['major'].apply(group_major_unified)
df_z['Timestamp'] = pd.to_datetime(df_z['Timestamp'], errors='coerce')
df_z['Reg_Day_Category'] = ((z_date - df_z['Timestamp']).dt.days).apply(get_reg_cat)

# Map projects from single string
for p in projects_list:
    df_z[p] = df_z['aiesec_project'].str.contains(p.replace(' ', ''), case=False, na=False).astype(int)

z_probs = model.predict_proba(df_z[features])[:, 1]

# ==========================================
# 4. PROCESS C1 DATA
# ==========================================
#print("Processing C1 Data...")
df_c1 = pd.read_csv('data/C1.csv')  # private, not included
c1_date = pd.Timestamp('2026-01-24')

df_c1['Degree'] = df_c1['*Сіздің білім деңгейіңіз: / Ваш уровень образования:*'].apply(map_degree_unified)
df_c1['University'] = df_c1['*Сіздің оқу орныңыз: / Ваше место обучения:*'].apply(map_uni_unified)
df_c1['Major_Group'] = df_c1['*Сіздің мамандығыңыз: / Ваша специальность:*'].apply(group_major_unified)
df_c1['Submitted At'] = pd.to_datetime(df_c1['Submitted At'], errors='coerce')
df_c1['Reg_Day_Category'] = ((c1_date - df_c1['Submitted At']).dt.days).apply(get_reg_cat)

# Map projects from single string
proj_col_c1 = '*AIESEC ұйымының қандай жобаларына қатысқансыз? / В каких проектах от AIESEC Вы принимали участие ранее? *'
for p in projects_list:
    df_c1[p] = df_c1[proj_col_c1].str.contains(p, case=False, na=False).astype(int)

c1_probs = model.predict_proba(df_c1[features])[:, 1]

# ==========================================
# 5. FINAL RESULTS
# ==========================================

print("Major group taken from: major")
def print_stats_c1(name, probs):
    print(f"\n--- {name} Results ---")
    print(f"Total Registered: {len(probs)}")
    print(f"Predicted Attendance (>50% of person likelihood): {int(sum(probs > 0.5))}")
    print(f"Predicted Attendance (>60% of person likelihood): {int(sum(probs > 0.6))}")
    #print(f"Estimated Attendance Rate: {round(np.mean(probs)*100, 2)}%")
    print(f"Attendance Rate (>60%): {round((sum(probs > 0.60)/len(df_c1))*100, 2)}%")
    print(f"Attendance Rate (>50%): {round((sum(probs > 0.5)/len(df_c1))*100, 2)}%")

def print_stats_zimran(name, probs):
    print(f"\n--- {name} Results ---")
    print(f"Total Registered: {len(probs)}")
    print(f"Predicted Attendance (>50%): {int(sum(probs > 0.5))}")
    print(f"Predicted Attendance (>60%): {int(sum(probs > 0.6))}")
    #print(f"Attendance Rate (>50%): {round(np.mean(probs)*100, 2)}%")
    print(f"Attendance Rate (>60%: {round((sum(probs > 0.6)/len(df_z))*100, 2)}%")
    print(f"Attendance Rate (>50%: {round((sum(probs > 0.5)/len(df_z))*100, 2)}%")


print_stats_c1("C1", c1_probs)
print_stats_zimran("ZIMRAN", z_probs)

In [ ]:
print("\nProcessing Zimran Data...")
print("Major group taken from: career interest")
df_z_1 = pd.read_csv('data/ZimranApril3.csv')  # private, not included
z_date = pd.Timestamp('2026-04-05')

df_z_1['Degree'] = df_z_1['year_of_study'].astype(str).apply(map_degree_unified)
df_z_1['University'] = df_z_1['university'].apply(map_uni_unified)
df_z_1['Major_Group'] = df_z_1['career_interest'].apply(group_major_unified)
df_z_1['Timestamp'] = pd.to_datetime(df_z_1['Timestamp'], errors='coerce')
df_z_1['Reg_Day_Category'] = ((z_date - df_z_1['Timestamp']).dt.days).apply(get_reg_cat)

# Map projects from single string
for p in projects_list:
    df_z_1[p] = df_z_1['aiesec_project'].str.contains(p.replace(' ', ''), case=False, na=False).astype(int)

z_probs_1 = model.predict_proba(df_z_1[features])[:, 1]

print_stats_zimran("ZIMRAN", z_probs_1)

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

# ==========================================
# 1. THE UPDATED UNIFIED MAJOR MAPPING
# ==========================================
def group_major_unified(major):
    m = str(major).lower().strip()
    if any(x in m for x in ['data', 'ai', 'аналит', 'интеллект', 'intelligence', 'bda', 'machine learning', 'ml']): 
        return 'Data Science & AI'
    if any(x in m for x in ['comput', 'soft', 'cs', 'программ', 'информ', 'it', 'swe', 'ict', 'developer', 'smart tech', 'web', 'ios']): 
        return 'Computer Science'
    if any(x in m for x in ['cyber', 'безопасн', 'security', 'secure']): 
        return 'IT & Cybersecurity'
    if any(x in m for x in ['engin', 'инженер', 'механ', 'строй', 'robot', 'робот', 'elce', 'ece', 'mae', 'cee', 'chme', 'mining', 'civil', 'automation', 'автоматиз', 'petroleum', 'geology']): 
        return 'Engineering'
    if any(x in m for x in ['econ', 'finan', 'accoun', 'эконом', 'финанс', 'учет', 'аудит', 'бухгалтер', 'finance']): 
        return 'Economics & Finance'
    if any(x in m for x in ['manag', 'marke', 'busin', 'менедж', 'маркет', 'бизнес', 'mba', 'bba', 'hr', 'logistics', 'логистика', 'tourism', 'туризм', 'smm', 'pr', 'предприн']): 
        return 'Business & Management'
    if any(x in m for x in ['polit', 'relat', 'socio', 'psir', 'journ', 'psych', 'право', 'юрисп', 'филол', 'лингв', 'literature', 'law', 'лингвистика', 'журналистика', 'психолог', 'педагог', 'teacher', 'мұғалім', 'wll', 'translation', 'аударма', 'media', 'archi', 'дизайн', 'design', 'filmmaking', 'history', 'ориентал', 'oriental']): 
        return 'Social Sciences & Humanities'
    if any(x in m for x in ['med', 'врач', 'health', 'pharma', 'биолог', 'biology', 'bio', 'медицина', 'стоматология', 'chemistry', 'хими', 'nusom', 'physics', 'физик']): 
        return 'Medicine & Science'
    if any(x in m for x in ['foundation', 'nufyp', 'undeclared', 'не определен']): 
        return 'Foundation'
    return 'Other'

def map_uni_unified(uni):
    uni = str(uni).lower()
    if any(x in uni for x in ['nazarbayev', 'назарбаев', 'nu']): return 'Nazarbayev University'
    if any(x in uni for x in ['astana it', 'aitu']): return 'Astana IT University'
    if any(x in uni for x in ['гумилев', 'enu', 'еуразия']): return 'Eurasian National University'
    if any(x in uni for x in ['narikbayev', 'kazguu', 'қазгу', 'mnu']): return 'Maqsut Narikbayev University'
    return 'Other'

def map_degree_unified(deg):
    deg = str(deg).lower()
    if '1' in deg: return '1st Year'
    if '2' in deg: return '2nd Year'
    if '3' in deg: return '3rd Year'
    if '4' in deg: return '4th Year'
    if 'master' in deg or 'магистрант' in deg: return "Master's"
    if 'phd' in deg or 'докторант' in deg: return "PhD"
    if any(x in deg for x in ['түлегі', 'выпускник', 'graduate']): return 'Graduate'
    return 'Other'

def get_reg_cat(days):
    if days <= 1: return "Last Minute"
    if days <= 3: return "Late"
    if days <= 7: return "Regular"
    return "Early Bird"

# ==========================================
# 2. PREPARE HIGGSFIELD DATA
# ==========================================
df = pd.read_csv('data/Higgsfield_Registrations.csv')  # private, not included
event_date = pd.Timestamp('2025-10-18')

# Apply Standard Mappings
df['Degree'] = df['Your education degree:'].apply(map_degree_unified)
df['University'] = df['Your place of study:'].apply(map_uni_unified)
df['Major_Group'] = df['Your major:'].apply(group_major_unified)

# Registration Timing
df['Submit Date (UTC)'] = pd.to_datetime(df['Submit Date (UTC)']).dt.tz_localize(None)
df['Reg_Day_Category'] = ((event_date - df['Submit Date (UTC)']).dt.days).apply(get_reg_cat)

# Past AIESEC Projects
projects = ['AIESEC Recruitment', 'Creative One', 'GO Exchanges', 'SheCodes', 'OZGE Forum', 'EmployMe']
for p in projects:
    df[p] = df[p].notna().astype(int)

# Features (Notice: NO AGE included)
features = ['Degree', 'University', 'Major_Group', 'Reg_Day_Category'] + projects
cat_features = ['Degree', 'University', 'Major_Group', 'Reg_Day_Category']

X = df[features]
y = df['CheckIn']

# ==========================================
# 3. TRAIN/TEST SPLIT AND ACCURACY CHECK
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=5,
    cat_features=cat_features,
    verbose=False
)

model.fit(X_train, y_train)

# Predict
preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

# Output Accuracy Results
print("--- HIGGSFIELD MODEL ACCURACY CHECK ---")
print(f"Overall Accuracy: {round(accuracy_score(y_test, preds)*100, 2)}%")
print(f"AUC Score: {round(roc_auc_score(y_test, probs), 4)}")
print("\n--- Detailed Report ---")
print(classification_report(y_test, preds))

# Feature Importance (Which factors matter most?)
print("\n--- What affects attendance most? ---")
importances = model.get_feature_importance()
for i in range(len(features)):
    print(f"{features[i]}: {round(importances[i], 2)}")